## 1. Imports and Configuration

In [11]:
import os
import sys
import random
import warnings
from typing import List, Optional

warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tifffile

from mmengine.config import Config
from mmengine.runner import Runner
from mmengine.model import BaseModule
from mmseg.registry import TRANSFORMS, MODELS
from mmseg.models.decode_heads.decode_head import BaseDecodeHead
from mmcv.transforms import BaseTransform

# Path setup
sys.path.insert(0, "/home/work/root/Meme_coin/ddc/mmsegmentation")
sys.path.insert(0, "/home/work/root/Meme_coin/ddc/mmsegmentation/mmseg")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.0.0a0+1767026
CUDA available: True


## 2. Global Configuration

In [12]:
# ==========================================
# Global Configuration
# ==========================================

# Random Seed
SEED = 42

# Paths
DATA_ROOT = '/home/work/root/Meme_coin/ddc/raw_data/extra_data'
WORK_DIR = '/home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f'

# Training Configuration
MAX_ITERS = 75000 # 300 epoch
VAL_INTERVAL = 2500 # 10 epoch
LOG_INTERVAL = 250 # 1 epoch
CHECKPOINT_INTERVAL = 2500
BATCH_SIZE = 16  # 256x256 이미지니까 배치 크게

# ⭐ LS30 Statistics (4-channel: RGBN)
LS30_MEAN = [10707.62, 10662.64, 9663.04, 15352.7]
LS30_STD = [1658.74, 1259.81, 1148.46, 3358.93]

os.makedirs(WORK_DIR, exist_ok=True)

print("✅ Configuration loaded")
print(f"   Work directory: {WORK_DIR}")
print(f"   Data root: {DATA_ROOT}")
print(f"   Max iterations: {MAX_ITERS:,}")

✅ Configuration loaded
   Work directory: /home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f
   Data root: /home/work/root/Meme_coin/ddc/raw_data/extra_data
   Max iterations: 75,000


## 3. Seed Fixing

In [14]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f"✅ Random seed fixed to {seed}")

set_seed(SEED)

✅ Random seed fixed to 42


## 4. Custom Transforms

In [15]:
# ==========================================
# Custom Transforms for LS30
# ==========================================

@TRANSFORMS.register_module()
class RemapLabels(BaseTransform):
    """라벨 리매핑: 10→1, 90→0"""
    def __init__(self, original_values=(10, 90), target_values=(1, 0)):
        self.original_values = original_values
        self.target_values = target_values
    
    def transform(self, results):
        if 'gt_seg_map' in results:
            seg_map = results['gt_seg_map']
            new_seg_map = seg_map.copy()
            for orig_val, target_val in zip(self.original_values, self.target_values):
                new_seg_map[seg_map == orig_val] = target_val
            results['gt_seg_map'] = new_seg_map
        return results


@TRANSFORMS.register_module()
class LoadLS30Image(BaseTransform):
    """LS30 4채널 TIFF 이미지 로드"""
    def transform(self, results):
        img_path = results['img_path']
        img = tifffile.imread(img_path).astype(np.float32)  # (256, 256, 4)
        img = np.transpose(img, (2, 0, 1))  # (4, 256, 256)
        
        results['img'] = img
        results['img_shape'] = img.shape[1:]
        results['ori_shape'] = img.shape[1:]
        return results


@TRANSFORMS.register_module()
class NormalizeLS30(BaseTransform):
    """LS30 정규화"""
    def __init__(self, mean, std, eps=1e-6):
        self.mean = np.array(mean, dtype=np.float32)[:, None, None]
        self.std = np.array(std, dtype=np.float32)[:, None, None]
        self.std = np.where(self.std < eps, 1.0, self.std)
    
    def transform(self, results):
        if 'img' in results:
            results['img'] = (results['img'] - self.mean) / self.std
        return results


@TRANSFORMS.register_module()
class PackLS30Inputs(BaseTransform):
    """LS30 데이터 패킹"""
    def __init__(self, meta_keys=('img_path', 'ori_shape', 'img_shape')):
        self.meta_keys = meta_keys
    
    def transform(self, results):
        from mmseg.structures import SegDataSample
        from mmengine.structures import PixelData
        
        inputs = torch.from_numpy(results['img'])
        sample = SegDataSample()
        
        if 'gt_seg_map' in results:
            seg = torch.from_numpy(results['gt_seg_map'][None, ...]).long()
            sample.gt_sem_seg = PixelData(data=seg)
        
        metainfo = {k: results[k] for k in self.meta_keys if k in results}
        sample.set_metainfo(metainfo)
        
        return dict(inputs=inputs, data_samples=sample)

print("✅ Custom transforms registered")

✅ Custom transforms registered


## 5. Model Architecture

In [16]:
# ==========================================
# Backbone with 1x1 Conv Adapter
# ==========================================

@MODELS.register_module()
class BackboneWithInputConv(BaseModule):
    """
    4-channel to 3-channel adapter for pretrained backbone
    RGB: Identity initialization
    NIR: Zero initialization
    """
    def __init__(self,
                 backbone: dict,
                 in_channels: int = 4,
                 out_channels: int = 3,
                 bias: bool = False,
                 init_cfg=None):
        super().__init__(init_cfg)
        assert out_channels == 3, "Pretrained backbone expects 3 channels"
        
        self.adapter = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=bias)
        self.backbone = MODELS.build(backbone)

        # RGB Identity initialization
        with torch.no_grad():
            w = self.adapter.weight
            w.zero_()
            w[0, 0, 0, 0] = 1.0  # R → R
            w[1, 1, 0, 0] = 1.0  # G → G
            w[2, 2, 0, 0] = 1.0  # B → B
            # NIR → 0 (already zero)
            
            if self.adapter.bias is not None:
                self.adapter.bias.zero_()
        
        print("✅ Adapter initialized: RGB identity, NIR zero")

    def forward(self, x):
        x = self.adapter(x)
        return self.backbone(x)

print("✅ BackboneWithInputConv registered")

✅ BackboneWithInputConv registered


## 6. Training Configuration

In [17]:
# ==========================================
# MMSegmentation Config for LS30 Pre-training
# ==========================================

cfg = Config(dict(
    default_scope='mmseg',
    backend_args=None,
    work_dir=WORK_DIR,
    
    # Data preprocessor
    data_preprocessor=dict(
        type='SegDataPreProcessor',
        bgr_to_rgb=False,
        pad_val=0,
        seg_pad_val=255,
        size=None,
        size_divisor=32
    ),
    
    # Model
    model=dict(
        type='EncoderDecoder',  # ⭐ 기본 EncoderDecoder (EnvMaps 불필요)
        data_preprocessor=dict(
            type='SegDataPreProcessor',
            bgr_to_rgb=False,
            pad_val=0,
            seg_pad_val=255,
            size=None,
            size_divisor=32
        ),
        
        # ⭐ 동일한 Backbone 구조!
        backbone=dict(
            type='BackboneWithInputConv',
            in_channels=4,
            out_channels=3,
            bias=False,
            backbone=dict(
                type='MixVisionTransformer',
                in_channels=3,
                embed_dims=64,
                num_stages=4,
                num_layers=[3, 6, 40, 3],  # ⭐ MiT-B5
                num_heads=[1, 2, 5, 8],
                patch_sizes=[7, 3, 3, 3],
                sr_ratios=[8, 4, 2, 1],
                out_indices=(0, 1, 2, 3),
                mlp_ratio=4,
                qkv_bias=True,
                drop_rate=0.0,
                attn_drop_rate=0.0,
                drop_path_rate=0.1,
                # ⭐ ImageNet pretrained weight로 시작
                init_cfg=dict(
                    type='Pretrained',
                    checkpoint='https://download.openmmlab.com/mmsegmentation/v0.5/pretrain/segformer/mit_b5_20220624-658746d9.pth'
                )
            )
        ),
        
        # ⭐ 간단한 SegFormer decode head
        decode_head=dict(
            type='SegformerHead',
            in_channels=[64, 128, 320, 512],  # MiT-B5 출력 채널
            in_index=[0, 1, 2, 3],
            channels=256,
            dropout_ratio=0.1,
            num_classes=2,  # Binary segmentation
            threshold=0.5,
            norm_cfg=dict(type='SyncBN', requires_grad=True),
            align_corners=False,
            out_channels=1,
            loss_decode=[
                dict(
                    type='CrossEntropyLoss',
                    use_sigmoid=True,
                    loss_weight=1.0,  # ← BCE 비중: 1
                    class_weight=[1.32],
                )
            ],
        ),
        
        train_cfg=dict(),
        test_cfg=dict(mode='whole'),
    ),
    
    # Data pipelines
    train_pipeline=[
        dict(type='LoadLS30Image'),
        dict(type='LoadAnnotations', imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(type='NormalizeLS30', mean=LS30_MEAN, std=LS30_STD),
        dict(type='PackLS30Inputs'),
    ],

    val_pipeline=[
        dict(type='LoadLS30Image'),
        dict(type='LoadAnnotations', imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(type='NormalizeLS30', mean=LS30_MEAN, std=LS30_STD),
        dict(type='PackLS30Inputs'),
    ],

    test_pipeline=[
        dict(type='LoadLS30Image'),
        dict(type='LoadAnnotations', imdecode_backend='tifffile'),
        dict(type='RemapLabels', original_values=(10, 90), target_values=(1, 0)),
        dict(type='NormalizeLS30', mean=LS30_MEAN, std=LS30_STD),
        dict(type='PackLS30Inputs'),
    ],
    
    # Dataloaders
    metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
    
    train_dataloader=dict(
        batch_size=BATCH_SIZE,
        num_workers=8,
        persistent_workers=True,
        sampler=dict(type='InfiniteSampler', shuffle=True),
        dataset=dict(
            type='BaseSegDataset',
            data_root=DATA_ROOT,
            metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
            data_prefix=dict(
                img_path='train/img/TS_LS30_LS30',
                seg_map_path='train/label/TL_LS30'
            ),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,
        ),
    ),
    
    val_dataloader=dict(
        batch_size=BATCH_SIZE,
        num_workers=4,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='BaseSegDataset',
            data_root=DATA_ROOT,
            metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
            data_prefix=dict(
                img_path='train/img/TS_LS30_LS30',  # validation split 없으면 train 사용
                seg_map_path='train/label/TL_LS30'
            ),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,
        ),
    ),
    
    test_dataloader=dict(
        batch_size=BATCH_SIZE,
        num_workers=4,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='BaseSegDataset',
            data_root=DATA_ROOT,
            metainfo=dict(classes=('background', 'industrial_complex'),palette=[[0, 0, 0], [0, 255, 0]], ),
            data_prefix=dict(
                img_path='train/img/TS_LS30_LS30',
                seg_map_path='train/label/TL_LS30'
            ),
            img_suffix='.tif',
            seg_map_suffix='.tif',
            pipeline=None,
        ),
    ),
    
    # Evaluators
    val_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    test_evaluator=dict(type='IoUMetric', iou_metrics=['mIoU']),
    
    # Optimizer
#     optim_wrapper=dict(
#         type='OptimWrapper',
#         optimizer=dict(type='AdamW', lr=1e-5, betas=(0.9, 0.999), weight_decay=0.01)
#     ),
    
    optim_wrapper = dict(
        type='OptimWrapper',
        optimizer=dict(
            type='AdamW',
            lr=6e-5,
            betas=(0.9, 0.999),
            weight_decay=0.01,
        ),
#         loss_scale='dynamic',
        paramwise_cfg=dict(
            # ✔ 모든 Norm 계열 (MiT + ENV 인코더 BN) WD=0
            norm_decay_mult=0.0,
            custom_keys={
                'pos_block': dict(decay_mult=0.0),  # MiT positional block WD=0
                'head': dict(lr_mult=10.0),         # 디코더 헤드 LR×10
            }
        )
    ),
    
    # Learning rate scheduler
    param_scheduler=[
        dict(type='LinearLR', start_factor=0.001, by_epoch=False, begin=0, end=1000),
        dict(type='PolyLR', begin=1000, end=MAX_ITERS, eta_min=1e-7, power=0.9, by_epoch=False)
    ],
    
    # Training loop
    train_cfg=dict(type='IterBasedTrainLoop', max_iters=MAX_ITERS, val_interval=VAL_INTERVAL),
    val_cfg=dict(type='ValLoop'),
    test_cfg=dict(type='TestLoop'),
    
    # Hooks
    default_hooks=dict(
        logger=dict(type='LoggerHook', interval=LOG_INTERVAL, log_metric_by_epoch=False),
        param_scheduler=dict(type='ParamSchedulerHook'),
        timer=dict(type='IterTimerHook'),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='SegVisualizationHook', draw=False),
        checkpoint=dict(
            type='CheckpointHook',
            by_epoch=False,
            interval=CHECKPOINT_INTERVAL,
            max_keep_ckpts=3,
            save_best='mIoU'
        )
    ),
    
    # Visualization
    vis_backends=[dict(type='LocalVisBackend', save_dir=WORK_DIR)],
    visualizer=dict(
        type='SegLocalVisualizer',
        vis_backends=[dict(type='LocalVisBackend', save_dir=WORK_DIR)],
        name='visualizer'
    ),
    
    # Environment
    env_cfg=dict(
        cudnn_benchmark=True,
        mp_cfg=dict(mp_start_method='fork', opencv_num_threads=0),
        dist_cfg=dict(backend='nccl')
    ),
    
    # Randomness
    randomness=dict(seed=SEED, deterministic=False, diff_rank_seed=False),
    
    launcher='none',
    log_level='INFO',
))

# Connect pipelines
cfg.train_dataloader.dataset.pipeline = cfg.train_pipeline
cfg.val_dataloader.dataset.pipeline = cfg.val_pipeline
cfg.test_dataloader.dataset.pipeline = cfg.test_pipeline

print("\n" + "="*60)
print("✅ Configuration Complete")
print("="*60)
print(f"Model: SegFormer MiT-B5 (ImageNet init)")
print(f"Data: LS30 4-channel satellite (256×256×4)")
print(f"Task: Binary segmentation")
print(f"Batch size: {BATCH_SIZE}")
print(f"Max Iterations: {MAX_ITERS:,}")
print(f"Validation Interval: {VAL_INTERVAL:,}")
print("="*60)


✅ Configuration Complete
Model: SegFormer MiT-B5 (ImageNet init)
Data: LS30 4-channel satellite (256×256×4)
Task: Binary segmentation
Batch size: 16
Max Iterations: 75,000
Validation Interval: 2,500


## 7. Training

In [18]:
# ==========================================
# Start Pre-training
# ==========================================

print("\n" + "="*60)
print("🚀 Starting LS30 Pre-training")
print("="*60)
print("목표: MiT-B5가 위성 이미지 패턴을 학습")
print("="*60)

runner = Runner.from_cfg(cfg)
runner.train()

print("\n" + "="*60)
print("✅ Pre-training Complete!")
print("="*60)
print(f"Results saved to: {WORK_DIR}")
print(f"Best checkpoint: {WORK_DIR}/best_mIoU_*.pth")
print("\n다음 단계: MiT-B5 weight 추출 (아래 셀 실행)")
print("="*60)


🚀 Starting LS30 Pre-training
목표: MiT-B5가 위성 이미지 패턴을 학습
11/17 20:47:06 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.10 (default, Nov 14 2022, 12:59:47) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0,1: NVIDIA H100 80GB HBM3
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.1, V12.1.66
    GCC: x86_64-linux-gnu-gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 2.0.0a0+1767026
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2021.1-Product Build 20201104 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash N/A)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.1
  - NVCC architecture flags: -g

✅ Adapter initialized: RGB identity, NIR zero
11/17 20:47:10 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
11/17 20:47:10 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:


11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.2.attn.norm.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.2.attn.norm.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.2.norm2.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.2.norm2.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.3.norm1.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.3.norm1.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.3.attn.norm.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.1.1.3.attn.norm.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- 

11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.6.attn.norm.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.6.attn.norm.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.6.norm2.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.6.norm2.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.7.norm1.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.7.norm1.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.7.attn.norm.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.7.attn.norm.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- 

11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.16.norm2.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.17.norm1.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.17.norm1.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.17.attn.norm.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.17.attn.norm.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.17.norm2.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.17.norm2.bias:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- backbone.backbone.layers.2.1.18.norm1.weight:weight_decay=0.0
11/17 20:47:13 - mmengine - INFO - paramwise_options -- 

...

11/18 00:35:30 - mmengine - INFO - Epoch(train) [1][51250/250]  base_lr: 2.1639e-05 lr: 2.1639e-05  eta: 1:42:12  time: 0.2547  data_time: 0.0233  memory: 8320  loss: 0.1085  decode.loss_ce: 0.1085  decode.acc_seg: 63.4178
11/18 00:36:34 - mmengine - INFO - Epoch(train) [1][51500/250]  base_lr: 2.1435e-05 lr: 2.1435e-05  eta: 1:41:07  time: 0.2555  data_time: 0.0232  memory: 8320  loss: 0.1081  decode.loss_ce: 0.1081  decode.acc_seg: 58.5333
11/18 00:37:38 - mmengine - INFO - Epoch(train) [1][51750/250]  base_lr: 2.1230e-05 lr: 2.1230e-05  eta: 1:40:03  time: 0.2548  data_time: 0.0234  memory: 8320  loss: 0.1084  decode.loss_ce: 0.1084  decode.acc_seg: 57.1928
11/18 00:38:42 - mmengine - INFO - Exp name: 20251117_204705
11/18 00:38:42 - mmengine - INFO - Epoch(train) [1][52000/250]  base_lr: 2.1026e-05 lr: 2.1026e-05  eta: 1:38:58  time: 0.2592  data_time: 0.0233  memory: 8320  loss: 0.1019  decode.loss_ce: 0.1019  decode.acc_seg: 56.7204
11/18 00:39:46 - mmengine - INFO - Epoch(train)

11/18 01:03:02 - mmengine - INFO - Saving checkpoint at 57500 iterations
11/18 01:03:25 - mmengine - INFO - Epoch(val) [0][250/250]    eta: 0:00:00  time: 0.0693  data_time: 0.0120  memory: 1713  
11/18 01:03:25 - mmengine - INFO - per class results:
11/18 01:03:25 - mmengine - INFO - 
+--------------------+-------+-------+
|       Class        |  IoU  |  Acc  |
+--------------------+-------+-------+
|     background     | 94.24 | 96.54 |
| industrial_complex |  90.9 | 96.03 |
+--------------------+-------+-------+
11/18 01:03:25 - mmengine - INFO - Epoch(val) [0][250/250]    aAcc: 96.3500  mIoU: 92.5700  mAcc: 96.2800  data_time: 0.0137  time: 0.0717
11/18 01:03:25 - mmengine - INFO - The previous best checkpoint /home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/best_mIoU_iter_52500.pth is removed
11/18 01:03:28 - mmengine - INFO - The best checkpoint with 92.5700 mIoU at 57500 iter is saved to best_mIoU_iter_57500.pth.
11/18 01:04:37 - mmengine - INFO - Epoch(train) [1][57750/

11/18 01:28:56 - mmengine - INFO - Epoch(train) [1][63250/250]  base_lr: 1.1533e-05 lr: 1.1533e-05  eta: 0:50:32  time: 0.2554  data_time: 0.0234  memory: 8320  loss: 0.1015  decode.loss_ce: 0.1015  decode.acc_seg: 66.7456
11/18 01:30:00 - mmengine - INFO - Epoch(train) [1][63500/250]  base_lr: 1.1314e-05 lr: 1.1314e-05  eta: 0:49:27  time: 0.2550  data_time: 0.0233  memory: 8320  loss: 0.0985  decode.loss_ce: 0.0985  decode.acc_seg: 55.5706
11/18 01:31:04 - mmengine - INFO - Epoch(train) [1][63750/250]  base_lr: 1.1094e-05 lr: 1.1094e-05  eta: 0:48:23  time: 0.2546  data_time: 0.0233  memory: 8320  loss: 0.0917  decode.loss_ce: 0.0917  decode.acc_seg: 60.7566
11/18 01:32:08 - mmengine - INFO - Exp name: 20251117_204705
11/18 01:32:08 - mmengine - INFO - Epoch(train) [1][64000/250]  base_lr: 1.0874e-05 lr: 1.0874e-05  eta: 0:47:18  time: 0.2548  data_time: 0.0236  memory: 8320  loss: 0.1009  decode.loss_ce: 0.1009  decode.acc_seg: 64.9716
11/18 01:33:11 - mmengine - INFO - Epoch(train)

11/18 01:57:45 - mmengine - INFO - Epoch(train) [1][69750/250]  base_lr: 5.6369e-06 lr: 5.6369e-06  eta: 0:22:34  time: 0.2549  data_time: 0.0234  memory: 8320  loss: 0.0970  decode.loss_ce: 0.0970  decode.acc_seg: 57.7886
11/18 01:58:49 - mmengine - INFO - Exp name: 20251117_204705
11/18 01:58:49 - mmengine - INFO - Epoch(train) [1][70000/250]  base_lr: 5.3990e-06 lr: 5.3990e-06  eta: 0:21:29  time: 0.2560  data_time: 0.0236  memory: 8320  loss: 0.0931  decode.loss_ce: 0.0931  decode.acc_seg: 59.5778
11/18 01:58:49 - mmengine - INFO - Saving checkpoint at 70000 iterations
11/18 01:59:16 - mmengine - INFO - Epoch(val) [0][250/250]    eta: 0:00:00  time: 0.0717  data_time: 0.0132  memory: 1713  
11/18 01:59:16 - mmengine - INFO - per class results:
11/18 01:59:16 - mmengine - INFO - 
+--------------------+-------+-------+
|       Class        |  IoU  |  Acc  |
+--------------------+-------+-------+
|     background     | 94.39 | 96.59 |
| industrial_complex | 91.13 |  96.2 |
+----------

11/18 02:21:35 - mmengine - INFO - The best checkpoint with 92.7700 mIoU at 75000 iter is saved to best_mIoU_iter_75000.pth.

✅ Pre-training Complete!
Results saved to: /home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f
Best checkpoint: /home/work/root/Meme_coin/ddc/work_dirs/pretrain_mitb5_f/best_mIoU_*.pth

다음 단계: MiT-B5 weight 추출 (아래 셀 실행)


## 8. Validation

In [20]:
# ==========================================
# Test Pre-trained Model
# ==========================================

if ckpt_files:
    print("Testing pre-trained model...")
    
    # Load config
    config_path = os.path.join(WORK_DIR, 'config.py')
    test_cfg = Config.fromfile(config_path)
    test_cfg.load_from = ckpt_files[0]
    
    # Run test
    test_runner = Runner.from_cfg(test_cfg)
    test_metrics = test_runner.test()
    
    print("\n" + "="*60)
    print("LS30 Pre-training Results")
    print("="*60)
    for key, value in test_metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    print("="*60)

Testing pre-trained model...
11/18 02:21:41 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.10 (default, Nov 14 2022, 12:59:47) [GCC 9.4.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0,1: NVIDIA H100 80GB HBM3
    CUDA_HOME: /usr/local/cuda
    NVCC: Cuda compilation tools, release 12.1, V12.1.66
    GCC: x86_64-linux-gnu-gcc (Ubuntu 9.4.0-1ubuntu1~20.04.1) 9.4.0
    PyTorch: 2.0.0a0+1767026
    PyTorch compiling details: PyTorch built with:
  - GCC 9.4
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2021.1-Product Build 20201104 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash N/A)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_52,code

✅ Adapter initialized: RGB identity, NIR zero
11/18 02:21:45 - mmengine - INFO - Distributed training is not used, all SyncBatchNorm (SyncBN) layers in the model will be automatically reverted to BatchNormXd layers if they are used.
11/18 02:21:45 - mmengine - INFO - Hooks will be executed in the following order:
before_run:
(VERY_HIGH   ) RuntimeInfoHook                    
(BELOW_NORMAL) LoggerHook                         
 -------------------- 
before_train:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(VERY_LOW    ) CheckpointHook                     
 -------------------- 
before_train_epoch:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
(NORMAL      ) DistSamplerSeedHook                
 -------------------- 
before_train_iter:
(VERY_HIGH   ) RuntimeInfoHook                    
(NORMAL      ) IterTimerHook                      
 -------------------- 
after_train_iter:
